# Stage 2 — Chunking + Embedding **v2** (clean rebuild)

Fixes 3 issues found in v1 baseline:
1. **De-duplication** — v1 had 1918 chunks but only 619 unique (3.1× accumulation from running `Chroma.from_documents()` 3 times into the same `persist_directory` without clearing)
2. **Tiny-chunk filter** — drop sections shorter than 50 chars (e.g. lone faculty headings, section title-only blocks)
3. **`chunk_id` metadata** — every chunk gets a unique `{source_file}_sec{idx}_off{start}` id for traceability

Also normalizes alias URLs to single `canonical_url` in metadata (URLs are noise to embeddings — keep ONLY in metadata for citation).

Embedding model fixed to `BAAI/bge-small-en-v1.5` (team decision).

## 0. Environment setup

In [ ]:
# Run once on Colab. Skip if the env already has these.
!pip install -q langchain langchain-community langchain-chroma langchain-huggingface chromadb sentence-transformers langchain-text-splitters

In [ ]:
# Clone repo (skip if you've already cloned)
import os
if not os.path.exists("RAG-based-Interactive-AI-for-MSADS-Website"):
    !git clone -q https://github.com/Yoki-SyZhang/RAG-based-Interactive-AI-for-MSADS-Website.git
%cd RAG-based-Interactive-AI-for-MSADS-Website/
!ls data/cleaned_sections/ | head

## 1. Config

In [ ]:
DB_PATH = "data/vector_store/MSADS_RAG_DB"
EMBED_MODEL = "BAAI/bge-small-en-v1.5"
CHUNK_SIZE = 600
CHUNK_OVERLAP = 50
MIN_SECTION_CHARS = 50    # drop sections shorter than this
SEPARATORS = ["\n## ", "\n\n", "\n", ".", " ", ""]

## 2. Wipe old Chroma DB (CRITICAL — prevents append accumulation)

In [ ]:
import shutil, os
if os.path.exists(DB_PATH):
    shutil.rmtree(DB_PATH)
    print(f"Wiped {DB_PATH}")
else:
    print(f"{DB_PATH} did not exist — fresh build")

## 3. Load Stage 1 cleaned sections + filter tiny

Each `data/cleaned_sections/page_*_cleaned.json` has:
- `url` (list of canonical + alias URLs)
- `canonical_url` (single str)
- `page_title`, `breadcrumbs`, `meta_description`, `Note`
- `structured_sections: [{section_index, text_markdown}, ...]`

We take ONLY `canonical_url` (alias URLs are redundant noise) and emit one Document per section with text length >= MIN_SECTION_CHARS.

In [ ]:
import json, glob
from langchain_core.documents import Document

raw_documents = []
total_sections = 0
filtered_tiny = 0

for fp in sorted(glob.glob("data/cleaned_sections/page_*_cleaned.json")):
    with open(fp, "r", encoding="utf-8") as f:
        data = json.load(f)

    canonical_url = data.get("canonical_url", "")
    title = data.get("page_title", "")
    breadcrumbs = data.get("breadcrumbs", "")
    source_file = data.get("source_file", fp.split("/")[-1])
    note = data.get("Note", "")

    for sec in data.get("structured_sections", []):
        total_sections += 1
        text = (sec.get("text_markdown") or "").strip()
        if len(text) < MIN_SECTION_CHARS:
            filtered_tiny += 1
            continue
        raw_documents.append(Document(
            page_content=text,
            metadata={
                "url": canonical_url,
                "title": title,
                "breadcrumbs": breadcrumbs,
                "source_file": source_file,
                "section_index": sec.get("section_index", -1),
                "note": note,
            }
        ))

print(f"sections seen:    {total_sections}")
print(f"sections kept:    {len(raw_documents)}  ({100*len(raw_documents)/max(total_sections,1):.1f}%)")
print(f"tiny filtered:    {filtered_tiny}  (<{MIN_SECTION_CHARS} chars)")

## 4. Chunk + add `chunk_id` + content-hash dedup

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
import hashlib

splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
    separators=SEPARATORS,
    add_start_index=True,
)
all_chunks = splitter.split_documents(raw_documents)

# Add chunk_id from source_file + section_index + start_index
for c in all_chunks:
    sf  = c.metadata.get("source_file", "")
    si  = c.metadata.get("section_index", -1)
    soi = c.metadata.get("start_index", 0)
    c.metadata["chunk_id"] = f"{sf}_sec{si}_off{soi}"

# Content-hash dedup (handles the page_148 / page_149 curriculum overlap and any other source dups)
seen_hashes = set()
deduped = []
for c in all_chunks:
    h = hashlib.md5(c.page_content.encode("utf-8")).hexdigest()
    if h in seen_hashes:
        continue
    seen_hashes.add(h)
    deduped.append(c)

print(f"chunks before dedup: {len(all_chunks)}")
print(f"chunks after dedup:  {len(deduped)}  ({100*(1 - len(deduped)/max(len(all_chunks),1)):.1f}% removed)")

# Length distribution
import statistics
lens = [len(c.page_content) for c in deduped]
print(f"chunk length min/median/mean/max: {min(lens)} / {int(statistics.median(lens))} / {int(statistics.mean(lens))} / {max(lens)}")

## 5. Embed + persist to Chroma

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

embeddings = HuggingFaceEmbeddings(model_name=EMBED_MODEL)
vector_db = Chroma.from_documents(
    documents=deduped,
    embedding=embeddings,
    persist_directory=DB_PATH,
)

print(f"\n✓ Built Chroma DB at {DB_PATH}")
print(f"  collection count: {vector_db._collection.count()}")

## 6. Audit — confirm clean build

In [ ]:
# Verify no duplicates remain in the persisted DB
res = vector_db._collection.get(include=["documents", "metadatas"])
docs_in_db = res["documents"]
print(f"persisted chunks: {len(docs_in_db)}")
print(f"unique content:   {len(set(docs_in_db))}")
assert len(docs_in_db) == len(set(docs_in_db)), "❌ duplicates leaked into DB!"
print("✓ no dups\n")

# URL distribution
from collections import Counter
url_counts = Counter(m.get("url") for m in res["metadatas"])
print("URLs covered:")
for u, n in sorted(url_counts.items(), key=lambda x: -x[1]):
    print(f"  {n:>4}× {u}")

## 7. Smoke-test retrieval

In [ ]:
retriever = vector_db.as_retriever(search_kwargs={"k": 3})

queries = [
    "What is the application deadline?",
    "Tell me about the capstone project",
    "Who teaches machine learning?",
    "What are the core courses?",
    "Difference between in-person and online program?",
]

for q in queries:
    print(f"\n>>> Q: {q}")
    results = retriever.invoke(q)
    for i, doc in enumerate(results):
        bc = doc.metadata.get("breadcrumbs", "")[:70]
        print(f"  #{i+1} [{bc}]")
        print(f"      {doc.page_content[:180].replace(chr(10), ' ')}")

## Next steps

After confirming this notebook runs clean:
1. Replace `scripts/Stage2_retrieval_system/chunking_and_embedding.ipynb` with this v2 (or commit alongside as `chunking_and_embedding_v2.ipynb`)
2. Commit the new `data/vector_store/MSADS_RAG_DB/` (~14 MB still — could move to git LFS later)
3. Stage 3 — build the RAG QA pipeline using this DB:
   - System instruction template (grounded-only, refuse out-of-scope, cite breadcrumb + URL)
   - 30-question eval test set (R@5 retrieval + LLM-as-judge response relevance)
